# HC3 Dataset Preprocessing Pipeline

This notebook prepares the HC3 (Human ChatGPT Comparison Corpus) dataset for binary classification.
The output is a clean `cleaned_data.csv` with exactly two columns: `answers` and `label`.

- `label = 0` → human-written answer  
- `label = 1` → ChatGPT-generated answer

**Tools used:** pandas, ast — no NLP libraries.

---
## Step 1 — Load and Inspect

We load the raw CSV and do a first-pass inspection: shape, null counts, duplicate counts,
and a head preview. This confirms the file loaded correctly and gives us a baseline
before any transformation.

In [1]:
import pandas as pd
import ast

df = pd.read_csv('hc3_all.csv')

print('=== Raw Dataset Inspection ===')
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print()
print('--- Null counts per column ---')
print(df.isnull().sum())
print()
print(f'Duplicate rows: {df.duplicated().sum()}')
print()
print('--- First 5 rows ---')
df.head()

=== Raw Dataset Inspection ===
Shape: (24322, 5)
Columns: ['id', 'question', 'human_answers', 'chatgpt_answers', 'source']

--- Null counts per column ---
id                 0
question           0
human_answers      0
chatgpt_answers    0
source             0
dtype: int64

Duplicate rows: 0

--- First 5 rows ---


,id,question,human_answers,chatgpt_answers,source
0,0,"Why is every book I hear about a "" NY Times # ...","['Basically there are many categories of "" Bes...",['There are many different best seller lists t...,reddit_eli5
1,1,"If salt is so bad for cars , why do we use it ...",['salt is good for not dying in car crashes an...,"[""Salt is used on roads to help melt ice and s...",reddit_eli5
2,2,Why do we still have SD TV channels when HD lo...,"[""The way it works is that old TV stations got...","[""There are a few reasons why we still have SD...",reddit_eli5
3,3,Why has nobody assassinated Kim Jong - un He i...,"[""You ca n't just go around assassinating the ...",['It is generally not acceptable or ethical to...,reddit_eli5
4,4,How was airplane technology able to advance so...,['Wanting to kill the shit out of Germans driv...,['After the Wright Brothers made the first pow...,reddit_eli5


---
## Step 2 — Drop Irrelevant Columns

We drop `id`, `question`, `source`, and any auto-generated `Unnamed` columns that pandas
adds when a CSV has a trailing comma or index column saved by mistake.
Only `human_answers` and `chatgpt_answers` are needed downstream.

In [2]:
cols_to_drop = [col for col in df.columns
                if 'Unnamed' in col or col in ['id', 'question', 'source']]

print(f'Dropping columns: {cols_to_drop}')
df.drop(columns=cols_to_drop, inplace=True)

print(f'Remaining columns: {df.columns.tolist()}')
print(f'Shape after drop: {df.shape}')
df.head()

Dropping columns: ['id', 'question', 'source']
Remaining columns: ['human_answers', 'chatgpt_answers']
Shape after drop: (24322, 2)


,human_answers,chatgpt_answers
0,"['Basically there are many categories of "" Bes...",['There are many different best seller lists t...
1,['salt is good for not dying in car crashes an...,"[""Salt is used on roads to help melt ice and s..."
2,"[""The way it works is that old TV stations got...","[""There are a few reasons why we still have SD..."
3,"[""You ca n't just go around assassinating the ...",['It is generally not acceptable or ethical to...
4,['Wanting to kill the shit out of Germans driv...,['After the Wright Brothers made the first pow...


---
## Step 3 — Build Human Dataframe (Explode)

The `human_answers` column stores multiple answers per question as a **serialised Python list
string**, e.g. `"['Answer one', 'Answer two']"`. We must:

1. Parse each string into a real Python list using `ast.literal_eval()` (safe, unlike `eval()`)
2. Fall back gracefully if parsing fails — wrap the raw string in a list so no row is silently lost
3. Use `explode()` so every individual human answer becomes its own row
4. Add `label = 0` to mark these as human-written

Without exploding, each row would contain a whole list, making character-length comparisons
meaningless and inflating the apparent length of human answers vs. AI answers.

In [3]:
def safe_parse(val):
    """Parse a serialised list string with ast.literal_eval; fall back to a single-item list."""
    try:
        result = ast.literal_eval(val)
        if isinstance(result, list):
            return result
        return [str(result)]
    except (ValueError, SyntaxError):
        return [str(val)]

human_df = df[['human_answers']].copy()
print(f'Human df shape before explode: {human_df.shape}')

human_df['human_answers'] = human_df['human_answers'].apply(safe_parse)
human_df = human_df.explode('human_answers')
human_df.rename(columns={'human_answers': 'answers'}, inplace=True)
human_df['label'] = 0
human_df = human_df.reset_index(drop=True)

print(f'Human df shape after explode:  {human_df.shape}')
print(f'Sample human answers:')
print(human_df['answers'].head(3).tolist())

Human df shape before explode: (24322, 1)
Human df shape after explode:  (24322, 2)
Sample human answers:
['Basically there are many categories of " Best Seller " . Replace " Best Seller " by something like " Oscars " and every " best seller " book is basically an " oscar - winning " book . May not have won the " Best film " , but even if you won the best director or best script , you \'re still an " oscar - winning " film . Same thing for best sellers . Also , IIRC the rankings change every week or something like that . Some you might not be best seller one week , but you may be the next week . I guess even if you do n\'t stay there for long , you still achieved the status . Hence , # 1 best seller .If you \'re hearing about it , it \'s because it was a very good or very well - publicized book ( or both ) , and almost every good or well - publicized book will be # 1 on the NY Times bestseller list for at least a little bit . Kindof like how almost every big or good movies are # 1 at t

---
## Step 4 — Build AI Dataframe (Explode)

Same approach for `chatgpt_answers`. Although the dataset typically stores a single AI answer
per row, it is still stored as a serialised list string, and some rows may contain multiple
AI answers. We parse and explode for consistency and to avoid any list objects leaking through.

In [4]:
ai_df = df[['chatgpt_answers']].copy()
print(f'AI df shape before explode: {ai_df.shape}')

ai_df['chatgpt_answers'] = ai_df['chatgpt_answers'].apply(safe_parse)
ai_df = ai_df.explode('chatgpt_answers')
ai_df.rename(columns={'chatgpt_answers': 'answers'}, inplace=True)
ai_df['label'] = 1
ai_df = ai_df.reset_index(drop=True)

print(f'AI df shape after explode:  {ai_df.shape}')
print(f'Sample AI answers:')
print(ai_df['answers'].head(3).tolist())

AI df shape before explode: (24322, 1)
AI df shape after explode:  (24322, 2)
Sample AI answers:
['There are many different best seller lists that are published by various organizations, and the New York Times is just one of them. The New York Times best seller list is a weekly list that ranks the best-selling books in the United States based on sales data from a number of different retailers. The list is published in the New York Times newspaper and is widely considered to be one of the most influential best seller lists in the book industry. \nIt\'s important to note that the New York Times best seller list is not the only best seller list out there, and there are many other lists that rank the top-selling books in different categories or in different countries. So it\'s possible that a book could be a best seller on one list but not on another. \nAdditionally, the term "best seller" is often used more broadly to refer to any book that is selling well, regardless of whether it is on 

---
## Step 5 — Combine and Clean

We concatenate the human and AI dataframes, then clean up:

- **Drop duplicates** — identical answer text with the same label is redundant
- **Drop nulls and empty strings** — any row where `answers` is NaN or blank after stripping
  would break downstream vectorisation

We print shape at each step to confirm nothing was lost unexpectedly, and check label balance.

In [5]:
df_combined = pd.concat([human_df, ai_df], ignore_index=True)
print(f'Shape after concat: {df_combined.shape}')

# Drop duplicates
before_dedup = df_combined.shape[0]
df_combined.drop_duplicates(inplace=True)
df_combined.reset_index(drop=True, inplace=True)
after_dedup = df_combined.shape[0]
print(f'Shape after drop_duplicates: {df_combined.shape}  (dropped {before_dedup - after_dedup} rows)')

# Drop nulls
before_null = df_combined.shape[0]
df_combined.dropna(subset=['answers'], inplace=True)
after_null = df_combined.shape[0]
print(f'Shape after dropna: {df_combined.shape}  (dropped {before_null - after_null} null rows)')

# Drop empty strings
before_empty = df_combined.shape[0]
df_combined = df_combined[df_combined['answers'].str.strip() != '']
df_combined.reset_index(drop=True, inplace=True)
after_empty = df_combined.shape[0]
print(f'Shape after empty-string drop: {df_combined.shape}  (dropped {before_empty - after_empty} rows)')

print()
print('--- Label distribution ---')
print(df_combined['label'].value_counts())

Shape after concat: (48644, 2)
Shape after drop_duplicates: (45750, 2)  (dropped 2894 rows)
Shape after dropna: (45749, 2)  (dropped 1 null rows)
Shape after empty-string drop: (45748, 2)  (dropped 1 rows)

--- Label distribution ---
label
1    23302
0    22446
Name: count, dtype: int64


---
## Step 6 — Lowercase and Basic Clean

We apply minimal text normalisation:
- Convert all answers to lowercase
- Strip leading/trailing whitespace

**No further cleaning** (no punctuation removal, no stopwords, no stemming).
The downstream TF-IDF + logistic regression pipeline handles the text as-is.

In [6]:
df_combined['answers'] = df_combined['answers'].str.lower().str.strip()

print('Lowercasing and whitespace stripping applied.')
print(f'Shape: {df_combined.shape}')
print()
print('Sample after normalisation:')
print(df_combined['answers'].head(3).tolist())

Lowercasing and whitespace stripping applied.
Shape: (45748, 2)

Sample after normalisation:
['basically there are many categories of " best seller " . replace " best seller " by something like " oscars " and every " best seller " book is basically an " oscar - winning " book . may not have won the " best film " , but even if you won the best director or best script , you \'re still an " oscar - winning " film . same thing for best sellers . also , iirc the rankings change every week or something like that . some you might not be best seller one week , but you may be the next week . i guess even if you do n\'t stay there for long , you still achieved the status . hence , # 1 best seller .if you \'re hearing about it , it \'s because it was a very good or very well - publicized book ( or both ) , and almost every good or well - publicized book will be # 1 on the ny times bestseller list for at least a little bit . kindof like how almost every big or good movies are # 1 at the box office

---
## Step 7 — Save Output

We assert that every value in `answers` is a plain string (not a list) — this confirms the
explode worked correctly. Then we save `cleaned_data.csv` with exactly two columns:
`answers` and `label`.

In [7]:
# Enforce only the two required columns
df_final = df_combined[['answers', 'label']].copy()

# Safety assertion: every answer must be a plain string, not a list
assert df_final['answers'].apply(lambda x: isinstance(x, str)).all(), \
    'ASSERTION FAILED: some answers are not plain strings — explode may not have worked!'
print('Assertion passed: all answers are plain strings.')

# Save
df_final.to_csv('cleaned_data.csv', index=False)
print(f'Saved cleaned_data.csv with shape: {df_final.shape}')
print(f'Columns: {df_final.columns.tolist()}')
print()
print('--- Head ---')
print(df_final.head())
print()
print('--- Tail ---')
print(df_final.tail())
print()
print(f'Final shape: {df_final.shape}')

Assertion passed: all answers are plain strings.
Saved cleaned_data.csv with shape: (45748, 2)
Columns: ['answers', 'label']

--- Head ---
                                             answers  label
0  basically there are many categories of " best ...      0
1  salt is good for not dying in car crashes and ...      0
2  the way it works is that old tv stations got a...      0
3  you ca n't just go around assassinating the le...      0
4  wanting to kill the shit out of germans drives...      0

--- Tail ---
                                                 answers  label
45743  it's not uncommon for blood pressure to fluctu...      1
45744  there are several possible causes of a painles...      1
45745  it is not appropriate for me to recommend a sp...      1
45746  it is not uncommon for people with rheumatoid ...      1
45747  it is not uncommon to experience back pain, es...      1

Final shape: (45748, 2)


---
## Step 8 — Sanity Check

A final audit of the cleaned dataset to confirm:

- Total rows and label balance
- Average answer length by label — **if human answers are still ~3× longer than AI answers,
  the explode did not work correctly** (because un-exploded human rows would still contain
  full list strings with multiple answers concatenated)
- Count and removal of suspiciously short answers (< 20 characters), which are likely
  parsing artefacts or empty placeholders

In [8]:
df_check = pd.read_csv('cleaned_data.csv')

print('===============================')
print('       FINAL SANITY CHECK      ')
print('===============================')

print(f'\nTotal rows: {len(df_check)}')

# Label distribution
print('\n--- Label distribution (counts) ---')
counts = df_check['label'].value_counts().sort_index()
print(counts)
print('\n--- Label distribution (percentages) ---')
print((counts / len(df_check) * 100).round(2).astype(str) + '%')

# Source distribution note
print('\n--- Source distribution ---')
print('(source column was dropped in Step 2; not available for distribution check)')

# Average character length by label
df_check['char_len'] = df_check['answers'].str.len()
print('\n--- Average character length by label ---')
avg_len = df_check.groupby('label')['char_len'].mean().round(1)
print(avg_len.rename({0: 'Human (label=0)', 1: 'AI (label=1)'})) 
ratio = avg_len[0] / avg_len[1] if avg_len[1] > 0 else float('inf')
print(f'Ratio human/AI: {ratio:.2f}x')
if ratio > 2.5:
    print('WARNING: Human answers are still much longer than AI answers.')
    print('         This suggests explode() may not have worked correctly.')
else:
    print('OK: Length ratio looks reasonable — explode appears to have worked.')

# Short answer audit
short_mask = df_check['answers'].str.strip().str.len() < 20
n_short = short_mask.sum()
print(f'\n--- Answers shorter than 20 characters: {n_short} ---')
if n_short > 0:
    print('Examples:')
    print(df_check.loc[short_mask, 'answers'].head(10).tolist())

# Drop short answers
before_short = len(df_check)
df_check = df_check[~short_mask].reset_index(drop=True)
after_short = len(df_check)
print(f'Dropped {before_short - after_short} rows with answers < 20 chars.')
print(f'Final shape after short-answer removal: {df_check.shape}')

# Overwrite cleaned_data.csv with the truly final version
df_check[['answers', 'label']].to_csv('cleaned_data.csv', index=False)
print('\ncleaned_data.csv overwritten with final clean version.')
print('\n===============================')
print('      PREPROCESSING COMPLETE   ')
print('===============================')

       FINAL SANITY CHECK      

Total rows: 45748

--- Label distribution (counts) ---
label
0    22446
1    23302
Name: count, dtype: int64

--- Label distribution (percentages) ---
label
0    49.06%
1    50.94%
Name: count, dtype: str

--- Source distribution ---
(source column was dropped in Step 2; not available for distribution check)

--- Average character length by label ---
label
Human (label=0)    1624.7
AI (label=1)       1143.7
Name: char_len, dtype: float64
Ratio human/AI: 1.42x
OK: Length ratio looks reasonable — explode appears to have worked.

--- Answers shorter than 20 characters: 4 ---
Examples:
['eric "e."', 'some ideas:', 'hello', 'test test 12']
Dropped 4 rows with answers < 20 chars.
Final shape after short-answer removal: (45744, 3)

cleaned_data.csv overwritten with final clean version.

      PREPROCESSING COMPLETE   
